#### Dataset: https://www.kaggle.com/datasets/absin7/zomato-bangalore-dataset

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/notebooks/midouazerty/restaurant-recommendation-system-using-ml/__notebook__.ipynb
/kaggle/input/notebooks/midouazerty/restaurant-recommendation-system-using-ml/__output__.json
/kaggle/input/notebooks/midouazerty/restaurant-recommendation-system-using-ml/__results__.html
/kaggle/input/notebooks/midouazerty/restaurant-recommendation-system-using-ml/__resultx__.html
/kaggle/input/notebooks/midouazerty/restaurant-recommendation-system-using-ml/custom.css
/kaggle/input/datasets/absin7/zomato-bangalore-dataset/dictionary_data.txt
/kaggle/input/datasets/absin7/zomato-bangalore-dataset/zomato.csv


### How the Restaurant Recommendation System works?

- Today, our world is a data-driven world.
- Data is used to create more efficient systems.
- Recommendation systems are a type of information filtering systems.
- Rec. Sys. improve the quality of search results and provide more relevant elements.
- RecSys are widely used to recommend movies, places to visit, items to buy, etc.

There are 2 types of recommendation systems:
1. Content-based filtering,
2. Collaborative filtering

We will start the task of Restaurant RecSys.

#### Library importing

In [22]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, r2_score

import warnings
warnings.filterwarnings('always')
warnings.filterwarnings('ignore')

import re
from nltk.corpus import stopwords

from sklearn.metrics.pairwise import linear_kernel
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

#### Read Dataset

In [23]:
zomato_real = pd.read_csv(
    "/kaggle/input/datasets/absin7/zomato-bangalore-dataset/zomato.csv"
    )
zomato_real.head()

,url,address,name,online_order,book_table,rate,votes,phone,location,rest_type,dish_liked,cuisines,approx_cost(for two people),reviews_list,menu_item,listed_in(type),listed_in(city)
0,https://www.zomato.com/bangalore/jalsa-banasha...,"942, 21st Main Road, 2nd Stage, Banashankari, ...",Jalsa,Yes,Yes,4.1/5,775,080 42297555\r\n+91 9743772233,Banashankari,Casual Dining,"Pasta, Lunch Buffet, Masala Papad, Paneer Laja...","North Indian, Mughlai, Chinese",800,"[('Rated 4.0', 'RATED\n A beautiful place to ...",[],Buffet,Banashankari
1,https://www.zomato.com/bangalore/spice-elephan...,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",Spice Elephant,Yes,No,4.1/5,787,080 41714161,Banashankari,Casual Dining,"Momos, Lunch Buffet, Chocolate Nirvana, Thai G...","Chinese, North Indian, Thai",800,"[('Rated 4.0', 'RATED\n Had been here for din...",[],Buffet,Banashankari
2,https://www.zomato.com/SanchurroBangalore?cont...,"1112, Next to KIMS Medical College, 17th Cross...",San Churro Cafe,Yes,No,3.8/5,918,+91 9663487993,Banashankari,"Cafe, Casual Dining","Churros, Cannelloni, Minestrone Soup, Hot Choc...","Cafe, Mexican, Italian",800,"[('Rated 3.0', ""RATED\n Ambience is not that ...",[],Buffet,Banashankari
3,https://www.zomato.com/bangalore/addhuri-udupi...,"1st Floor, Annakuteera, 3rd Stage, Banashankar...",Addhuri Udupi Bhojana,No,No,3.7/5,88,+91 9620009302,Banashankari,Quick Bites,Masala Dosa,"South Indian, North Indian",300,"[('Rated 4.0', ""RATED\n Great food and proper...",[],Buffet,Banashankari
4,https://www.zomato.com/bangalore/grand-village...,"10, 3rd Floor, Lakshmi Associates, Gandhi Baza...",Grand Village,No,No,3.8/5,166,+91 8026612447\r\n+91 9901210005,Basavanagudi,Casual Dining,"Panipuri, Gol Gappe","North Indian, Rajasthani",600,"[('Rated 4.0', 'RATED\n Very good restaurant ...",[],Buffet,Banashankari


#### Data Cleaning & Feature Engineering

1. Deleting Unnecessary Columns
2. Removing the Duplicates
3. Remove the NaN values from the dataset
4. Changing the column names
5. Data Transformations
6. Data Cleaning
7. Adjust the Column Names

##### 1. Deleting Unnnecessary Columns

In [24]:
zomato = zomato_real.drop(["url", "dish_liked", "phone"], axis=1)
zomato.columns

Index(['address', 'name', 'online_order', 'book_table', 'rate', 'votes',
       'location', 'rest_type', 'cuisines', 'approx_cost(for two people)',
       'reviews_list', 'menu_item', 'listed_in(type)', 'listed_in(city)'],
      dtype='object')

##### 2. Removing the Duplicates

In [25]:
zomato.duplicated().sum()
zomato.drop_duplicates(inplace=True)

##### 3. Remove the Nan values

In [26]:
zomato.isnull().sum()
zomato.dropna(how='any', inplace=True)

##### 4. Changing the column names

In [27]:
zomato = zomato.rename(columns={
    'approx_cost(for two people)': 'cost',
    'listed_in(type)': 'type',
    'listed_in(city)': 'city'
})

zomato.columns

Index(['address', 'name', 'online_order', 'book_table', 'rate', 'votes',
       'location', 'rest_type', 'cuisines', 'cost', 'reviews_list',
       'menu_item', 'type', 'city'],
      dtype='object')

##### 5. Transformation

In [28]:
# 'cost' column
# 1. Sütundaki tüm değerleri metne çevirip, virgülleri kaldırıyoruz
zomato['cost'] = zomato['cost'].astype(str).str.replace(',', '', regex=False)

# 2. Varsa noktalı virgülleri noktaya çeviriyoruz
zomato['cost'] = zomato['cost'].str.replace(';', '.', regex=False)

# 3. Sayıya çevrilemeyen hatalı veya boş metinleri NaN (boş veri) yapmak için pd.to_numeric kullanıyoruz
zomato['cost'] = pd.to_numeric(zomato['cost'], errors='coerce')

##### 6. Data Cleaning

In [29]:
zomato = zomato.loc[zomato.rate != 'NEW']
zomato = zomato.loc[zomato.rate != '-'].reset_index(drop=True)

zomato.rate = zomato.rate.astype(str).apply(lambda x: x.replace('/5', '')).str.strip()
zomato.rate = zomato.rate.astype('float')

##### 7. Adjust the Column Names

In [30]:
zomato.name = zomato.name.apply(lambda x:x.title())
zomato.online_order.replace(('Yes','No'),(True, False),inplace=True)
zomato.book_table.replace(('Yes','No'),(True, False),inplace=True)

#### Compute the Mean Rating

In [31]:
restaurants = list(zomato['name'].unique())
zomato['Mean Rating'] = 0

for i in range(len(restaurants)):
    zomato['Mean Rating'][zomato[
    'name'] == restaurants[i]] = zomato['rate'][zomato[
    'name'] == restaurants[i]].mean()    

In [32]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(1,5))
zomato[['Mean Rating']] = scaler.fit_transform(zomato[['Mean Rating']]).round(2)

#### Text Proprocessing

Steps:
- Lower casing
- Removal of Punctuations
- Removal of Stopwords
- Removal of URLs
- Spelling correction

In [33]:
# Lower casing
zomato['reviews_list'] = zomato['reviews_list'].str.lower()

In [34]:
# Remove punctuations
import string
PUNCT_TO_REMOVE = string.punctuation
print("punctuations: ", PUNCT_TO_REMOVE)

def remove_punctuations(text):
    """custom function to remove the puncs"""
    return text.translate(str.maketrans('', '', PUNCT_TO_REMOVE))

zomato["reviews_list"] = zomato["reviews_list"].apply(
    lambda text: remove_punctuations(text)
)

punctuations:  !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~


In [35]:
# remove stopwords
from nltk.corpus import stopwords
STOPWORDS = set(stopwords.words('english'))
print('English Stopwors: ', STOPWORDS)

def remove_stopwords(text):
    """removes the stopwords"""
    return "".join([word for word in str(text).split() if word not in STOPWORDS])

zomato["reviews_list"] = zomato["reviews_list"].apply(
    lambda text: remove_stopwords(text)
)

English Stopwors:  {'than', "couldn't", 'me', 'until', 'some', 'once', "that'll", 'hers', 'what', 'shan', "i'd", 'doesn', "they've", 'she', 'wouldn', "you'll", 'couldn', 'myself', "needn't", 'the', 'aren', 'wasn', "it'd", 'am', "haven't", 'not', 'these', 'few', 're', 'into', 'mustn', 'which', 'a', 'haven', "i'll", 'ma', 'm', "you're", 'then', "we'd", 'against', 'most', 'won', 'there', 'after', 'has', 's', 'weren', 'having', 'out', 'hadn', 'this', "hadn't", 'only', 'more', "they'll", "we'll", 'their', 'them', 'where', 'below', 'him', 'who', 'about', 'have', 'needn', 'no', 'when', 'ain', 'during', "weren't", 'd', "won't", 'but', 'nor', 'y', 'those', 'between', 'had', 'in', 'other', 'while', 'himself', 'were', 'yourselves', "it's", "wouldn't", 'don', "they're", 'of', 'theirs', 'any', 'that', 'or', "we're", 'if', 'they', "you'd", 'being', 'why', 'ours', "she'll", "shouldn't", 'just', 'you', 'off', 'now', 'his', "didn't", 'how', 'been', 'too', 'themselves', "aren't", 'i', 'for', 'down', 'he

In [36]:
# removal of URLs
def remove_urls(text):
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

zomato["reviews_list"] = zomato["reviews_list"].apply(
    lambda text: remove_urls(text)
)

zomato[['reviews_list', 'cuisines']].sample(5)

,reviews_list,cuisines
40812,rated40ratednorderedfoodalwayswonderfulexperie...,"Andhra, Biryani"
6394,rated40ratednbestplacebreakfastsweetpeaskachor...,"Street Food, North Indian, Fast Food"
11271,rated50ratednexcellentserviceexcellentfoodnone...,"North Indian, Chinese, Seafood, South Indian"
38019,,Burmese
16243,rated40ratednplacelocatedjaynagartalkingrestau...,"South Indian, North Indian, Chinese, Kebab"


##### Restaurant Names

In [37]:
restaurant_names = list(zomato['name'].unique())

def get_top_words(column, top_nof_words, nof_words):
    vec = CountVectorizer(ngram_range=nof_words, stop_words='english')
    bag_of_words = vec.fit_transform(column)
    sum_words = bag_of_words.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)

    return words_freq[:top_nof_words]

zomato=zomato.drop(['address','rest_type', 'type', 'menu_item', 'votes'],axis=1)
import pandas

# Randomly sample 60% of your dataframe
df_percent = zomato.sample(frac=0.5)

#### TF-IDF Vectorization
- TF-IDF: Term Frequency-Inverse Document Frequency
- TF-IDF vectors for each document.
- TF-IDF will give you a matrix.
- In the matrix, each columns(each restaurants) represents a word in the general vocabulary.(all words that appear in at least one document)
- TF-IDF is the statistical method of assesing the meaning of a word in a given document.

Now, we will use the TF-IDF vectorization on the dataset:

In [38]:
df_percent.set_index('name', inplace=True)
indices = pd.Series(df_percent.index)

# creating tf-idf matrix
tf_idf = TfidfVectorizer(
    analyzer='word', ngram_range=(1,2), min_df=0., stop_words='english'
)

tf_idf_matrix = tf_idf.fit_transform(df_percent['reviews_list'])

cosine_similarities = linear_kernel(tf_idf_matrix, tf_idf_matrix)

In [41]:
def recommend(name, cosine_similarities=cosine_similarities):
    # Find the index of the restaurant entered
    idx = indices[indices == name].index[0]
    
    # Find the restaurants with a similar cosine-sim value and order them in descending order
    score_series = pd.Series(cosine_similarities[idx]).sort_values(ascending=False)
    
    # Extract top 30 similar restaurant indices (excluding the restaurant itself at index 0)
    top30_indices = score_series.iloc[1:31].index
    
    # Get the names of the top 30 restaurants
    recommend_restaurant = [df_percent.index[i] for i in top30_indices]
    
    # Create the top 30 similar restaurants dataframe efficiently using .loc
    df_new = df_percent.loc[recommend_restaurant, ['cuisines', 'Mean Rating', 'cost']]
    
    # Drop duplicate rows and sort only the top 10 by the highest rating
    df_new = df_new.drop_duplicates(subset=['cuisines', 'Mean Rating', 'cost'])
    df_new = df_new.sort_values(by='Mean Rating', ascending=False).head(10)
    
    print('TOP %s RESTAURANTS LIKE %s WITH SIMILAR REVIEWS: ' % (str(len(df_new)), name))
    
    return df_new

In [42]:
recommend('Pai Vihar')

TOP 10 RESTAURANTS LIKE Pai Vihar WITH SIMILAR REVIEWS: 


,cuisines,Mean Rating,cost
name,,,
The Pancake Story,"Desserts, Beverages",4.61,300
Koramangala Social,"American, North Indian, Chinese, Finger Food",4.61,1500
Chianti,Italian,4.59,1500
Mainland China,Chinese,4.48,1700
Arbor Brewing Company,"American, Continental, Salad",4.48,2000
Arbor Brewing Company,"American, Continental",4.48,2000
Mainland China,"Chinese, Momos",4.48,1100
Mainland China,"Chinese, Momos",4.48,1700
"Lush - Renaissance Hotel, Bengaluru","North Indian, Continental",4.48,1200
